In [45]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler

import joblib

In [46]:
commodity_files = {
    "Onion":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv",
    "Rice":        "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv",
    "Wheat":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv",
    "Cabbage":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv",
    "Cauliflower": "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv",
    "Brinjal":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv"
}



In [47]:
for commodity, filepath in commodity_files.items():
    df = pd.read_csv(filepath, skiprows=1)
    price_series = pd.to_numeric(df['Modal Price'], errors='coerce').dropna().values
    print(f"{commodity}: min=₹{price_series.min():.0f}  max=₹{price_series.max():.0f}  mean=₹{price_series.mean():.0f}  rows={len(price_series)}")

Onion: min=₹10  max=₹999  mean=₹727  rows=10381
Rice: min=₹36  max=₹800  mean=₹293  rows=3
Wheat: min=₹26  max=₹805  mean=₹319  rows=7
Cabbage: min=₹200  max=₹990  mean=₹681  rows=13946
Cauliflower: min=₹17  max=₹993  mean=₹702  rows=7875
Brinjal: min=₹30  max=₹998  mean=₹753  rows=4585


In [48]:
for commodity, filepath in commodity_files.items():
    df = pd.read_csv(filepath, skiprows=1)
    print(f"\n{'='*50}")
    print(f"{commodity} — Total rows: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    if 'State' in df.columns:
        print(f"Top 3 States:\n{df['State'].value_counts().head(3)}")
    if 'Market' in df.columns:
        print(f"Top 3 Markets:\n{df['Market'].value_counts().head(3)}")


Onion — Total rows: 107986
Columns: ['State', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']
Top 3 States:
State
Tamil Nadu       30131
Uttar Pradesh    10419
Keralam           9601
Name: count, dtype: int64
Top 3 Markets:
Market
Bangalore APMC       498
Indore APMC          445
Koduvayoor Market    348
Name: count, dtype: int64

Rice — Total rows: 25170
Columns: ['State', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']
Top 3 States:
State
West Bengal      9668
Uttar Pradesh    8357
Maharashtra      2083
Name: count, dtype: int64
Top 3 Markets:
Market
Kalyani APMC              410
Kalipur APMC              363
Baruipur(Canning) APMC    360
Name: count, dtype: int64

Wheat — Total rows: 118477
Columns: ['State', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 

In [49]:
import os
for root, dirs, files in os.walk('/Users/mukulbhamgare/Desktop/CommodityPricePrediction'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Price_Agriculture_commodities_Week.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/commodity_prices.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/cleaned_commodity_data.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv


In [50]:
class LSTMModel(nn.Module):

    def __init__(self):

        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(

            input_size=1,

            hidden_size=50,

            batch_first=True

        )

        self.fc = nn.Linear(

            50,

            1

        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = self.fc(out[:, -1, :])

        return out

In [51]:
import os
result = os.popen('find /Users/mukulbhamgare/Desktop/CommodityPricePrediction -name "*.csv"').read()
print(result)

/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Price_Agriculture_commodities_Week.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/commodity_prices.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/cleaned_commodity_data.csv
/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv



In [52]:
for commodity, filepath in commodity_files.items():
    df = pd.read_csv(filepath, skiprows=1)
    df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
    price_series = pd.to_numeric(df['Modal Price'], errors='coerce').dropna().values
    print(f"{commodity}: min=₹{price_series.min():.0f}, max=₹{price_series.max():.0f}, mean=₹{price_series.mean():.0f}, count={len(price_series)}")

Onion: min=₹10, max=₹999, mean=₹727, count=10381
Rice: min=₹36, max=₹800, mean=₹293, count=3
Wheat: min=₹26, max=₹805, mean=₹319, count=7
Cabbage: min=₹200, max=₹990, mean=₹681, count=13946
Cauliflower: min=₹17, max=₹993, mean=₹702, count=7875
Brinjal: min=₹30, max=₹998, mean=₹753, count=4585


In [53]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv", skiprows=1)
print(df['Market'].value_counts().head(10))

Market
Kalyani APMC                  410
Kalipur APMC                  363
Baruipur(Canning) APMC        360
Bishalgarh APMC               353
Kalburgi APMC                 351
Karanjia APMC                 344
Guskara APMC                  317
Ramkrishanpur(Howrah) APMC    310
Kalna APMC                    305
Uluberia APMC                 303
Name: count, dtype: int64


In [54]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

commodity_files = {
    "Onion":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv",
    "Rice":        "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv",
    "Wheat":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv",
    "Cabbage":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv",
    "Cauliflower": "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv",
    "Brinjal":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv",
}

MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"

class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=50, batch_first=True)
        self.fc = nn.Linear(50, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

for commodity, filepath in commodity_files.items():
    print("Loop started")
    print(f"\n{'='*50}")
    df = pd.read_csv(filepath, skiprows=1)
    
    df = df[
    df['Price Unit'].str.contains('Quintal', na=False)
    ]
    print(df.head())

    
    
    print(f"Training: {commodity}")
    df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
    df = df.sort_values('Price Date')
    df['Modal Price'] = (
    df['Modal Price']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.strip()
    )

    price_series = pd.to_numeric(
    df['Modal Price'],
    errors='coerce'
    ).dropna().values

    price_series = price_series[
    price_series > 0
    ]
    print(f"Rows after filter: {len(price_series)}")

    if len(price_series) < 10:
        print(f"⚠ Not enough data, skipping.")
        continue

    print(f"Price range: ₹{price_series.min():.0f} – ₹{price_series.max():.0f}  |  Mean: ₹{price_series.mean():.0f}")

    # Scale
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(price_series.reshape(-1, 1))

    # Sequences
    SEQ_LEN = 10
    X, y = [], []
    for i in range(SEQ_LEN, len(scaled_data)):
        X.append(scaled_data[i-SEQ_LEN:i])
        y.append(scaled_data[i])

    X = torch.tensor(np.array(X, dtype=np.float32)).view(-1, SEQ_LEN, 1)
    y = torch.tensor(np.array(y, dtype=np.float32))

    # Train
    model = LSTMModel()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(300):
        model.train()
        outputs = model(X)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/100 — Loss: {loss.item():.6f}")

    # Save
    torch.save(model.state_dict(), f"{MODEL_DIR}{commodity.lower()}_lstm_model.pth")
    joblib.dump(scaler, f"{MODEL_DIR}{commodity.lower()}_scaler.save")
    print(f"✅ {commodity} done!")

print("\n🎉 All models retrained!")

Loop started

            State District        Market Commodity Group Commodity Variety  \
0  Andhra Pradesh  Kurnool  Kurnool APMC      Vegetables     Onion   Local   
1  Andhra Pradesh  Kurnool  Kurnool APMC      Vegetables     Onion   Local   
2  Andhra Pradesh  Kurnool  Kurnool APMC      Vegetables     Onion   Local   
3  Andhra Pradesh  Kurnool  Kurnool APMC      Vegetables     Onion   Local   
4  Andhra Pradesh  Kurnool  Kurnool APMC      Vegetables     Onion   Local   

   Grade Min Price Max Price Modal Price   Price Unit  Price Date  
0    FAQ    325.00    771.00      771.00  Rs./Quintal  28-04-2026  
1    FAQ    580.00    671.00      671.00  Rs./Quintal  27-04-2026  
2    FAQ    610.00    719.00      610.00  Rs./Quintal  25-04-2026  
3  Local    650.00    650.00      650.00  Rs./Quintal  24-04-2026  
4    FAQ    415.00  1,021.00      515.00  Rs./Quintal  23-04-2026  
Training: Onion
Rows after filter: 107986
Price range: ₹10 – ₹8100  |  Mean: ₹2056
  Epoch 20/100 — Loss: 0.0

In [55]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

In [57]:
df = pd.read_csv(
    "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv",
    skiprows=1
)

print(df.head())

            State District         Market Commodity Group Commodity Variety  \
0  Andhra Pradesh  Krishna  Tiruvuru APMC         Cereals      Rice    Sona   
1  Andhra Pradesh  Krishna  Tiruvuru APMC         Cereals      Rice    Fine   
2  Andhra Pradesh  Krishna  Tiruvuru APMC         Cereals      Rice    Sona   
3  Andhra Pradesh  Krishna  Tiruvuru APMC         Cereals      Rice    Sona   
4  Andhra Pradesh  Krishna  Tiruvuru APMC         Cereals      Rice    Sona   

   Grade Min Price Max Price Modal Price   Price Unit  Price Date  
0  Local  4,600.00  4,600.00    4,600.00  Rs./Quintal  10-12-2025  
1  Local  4,700.00  4,700.00    4,700.00  Rs./Quintal  09-12-2025  
2  Local  4,800.00  4,800.00    4,800.00  Rs./Quintal  08-12-2025  
3  Local  4,800.00  4,800.00    4,800.00  Rs./Quintal  02-12-2025  
4  Local  4,800.00  4,800.00    4,800.00  Rs./Quintal  29-11-2025  


In [58]:
df.columns = [

    'State',
    'District',
    'Market',
    'Commodity Group',
    'Commodity',
    'Variety',
    'Grade',
    'Min Price',
    'Max Price',
    'Modal Price',
    'Price Unit',
    'Price Date'

]

print(df.columns)

Index(['State', 'District', 'Market', 'Commodity Group', 'Commodity',
       'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price',
       'Price Unit', 'Price Date'],
      dtype='str')


In [59]:
df['Modal Price'] = (

    df['Modal Price']
    .astype(str)
    .str.replace(',', '', regex=False)

)

In [60]:
df = df[
    df['Price Unit'] == 'Rs./Quintal'
]

print(len(df))

25170


In [61]:
price_series = pd.to_numeric(

    df['Modal Price'],
    errors='coerce'

).dropna().values

print(price_series[:10])

[4600. 4700. 4800. 4800. 4800. 4800. 4800. 4700. 4700. 4700.]


In [62]:
price_series = price_series[

    (price_series > 1500) &
    (price_series < 7000)

]

print("Min:", price_series.min())

print("Max:", price_series.max())

print("Rows:", len(price_series))

Min: 1520.0
Max: 6998.11
Rows: 24644


In [63]:
scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(
    price_series.reshape(-1,1)
)

print(scaled_data[:5])

[[0.56223771]
 [0.58049218]
 [0.59874665]
 [0.59874665]
 [0.59874665]]


In [64]:
SEQ_LEN = 10

X = []
y = []

for i in range(SEQ_LEN, len(scaled_data)):

    X.append(
        scaled_data[i-SEQ_LEN:i]
    )

    y.append(
        scaled_data[i]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(24634, 10, 1)
(24634, 1)


In [65]:
import torch

X = torch.tensor(
    X,
    dtype=torch.float32
)

y = torch.tensor(
    y,
    dtype=torch.float32
)

print(X.shape)
print(y.shape)

torch.Size([24634, 10, 1])
torch.Size([24634, 1])


In [66]:
import torch.nn as nn

class LSTMModel(nn.Module):

    def __init__(self):

        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            batch_first=True
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = self.fc(out[:, -1, :])

        return out

In [67]:
model = LSTMModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

LSTMModel(
  (lstm): LSTM(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [68]:
for epoch in range(150):

    output = model(X)

    loss = criterion(
        output,
        y
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:

        print(
            f"Epoch {epoch+1}/150 "
            f"Loss: {loss.item():.6f}"
        )

Epoch 20/150 Loss: 0.027424
Epoch 40/150 Loss: 0.023705
Epoch 60/150 Loss: 0.018837
Epoch 80/150 Loss: 0.010116
Epoch 100/150 Loss: 0.009578
Epoch 120/150 Loss: 0.009429
Epoch 140/150 Loss: 0.009329


In [69]:
torch.save(

    model.state_dict(),

    "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/rice_lstm_model.pth"

)

joblib.dump(

    scaler,

    "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/rice_scaler.save"

)

print("✅ Rice model saved successfully!")

✅ Rice model saved successfully!


In [70]:
import joblib

torch.save(
    model.state_dict(),
    "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/rice_lstm_model.pth"
)

joblib.dump(
    scaler,
    "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/rice_scaler.save"
)

print("✅ Rice model saved successfully!")

✅ Rice model saved successfully!


In [71]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPrediction/app/data/Rice.csv", skiprows=1)
print(df['Price Unit'].value_counts())
print("\n")
quintal_df = df[df['Price Unit'] == 'Quintal']
print(f"Quintal rows: {len(quintal_df)}")
print(quintal_df['Market'].value_counts().head(10))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/mukulbhamgare/Desktop/CommodityPrediction/app/data/Rice.csv'

In [72]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv", skiprows=1)
print(df['Price Unit'].value_counts())
print("\n")
quintal_df = df[df['Price Unit'] == 'Quintal']
print(f"Quintal rows: {len(quintal_df)}")
print(quintal_df['Market'].value_counts().head(10))

Price Unit
Rs./Quintal    25170
Name: count, dtype: int64


Quintal rows: 0
Series([], Name: count, dtype: int64)


In [73]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv", skiprows=1)

# Top market dhundho with actual data
top_market = df['Market'].value_counts().index[0]
print(f"Top market: '{top_market}'")
print(f"Length: {len(top_market)}")
print(f"Repr: {repr(top_market)}")

# Filter test
df_test = df[df['Market'] == top_market]
print(f"Rows: {len(df_test)}")

Top market: 'Kalyani APMC'
Length: 12
Repr: 'Kalyani APMC'
Rows: 410


In [80]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv", skiprows=1)
top_market = df['Market'].value_counts().index[0]
df = df[df['Market'] == top_market].copy()
df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
df = df.sort_values('Price Date')

# FIX: comma remove karo pehle
price_series = df['Modal Price'].astype(str).str.replace(',', '').pipe(pd.to_numeric, errors='coerce').dropna().values

print(f"Rows: {len(price_series)}")
print(f"Price range: ₹{price_series.min():.0f} – ₹{price_series.max():.0f} | Mean: ₹{price_series.mean():.0f}")

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(price_series.reshape(-1, 1))

SEQ_LEN = 3
X, y = [], []
for i in range(SEQ_LEN, len(scaled_data)):
    X.append(scaled_data[i-SEQ_LEN:i])
    y.append(scaled_data[i])

X = torch.tensor(np.array(X, dtype=np.float32)).view(-1, SEQ_LEN, 1)
y = torch.tensor(np.array(y, dtype=np.float32))

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    outputs = model(X)
    loss = criterion(outputs, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1}/100 — Loss: {loss.item():.6f}")

MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"
torch.save(model.state_dict(), f"{MODEL_DIR}rice_lstm_model.pth")
joblib.dump(scaler, f"{MODEL_DIR}rice_scaler.save")
print("✅ Rice retrained!")

Rows: 410
Price range: ₹3300 – ₹4900 | Mean: ₹4316
  Epoch 20/100 — Loss: 0.179617
  Epoch 40/100 — Loss: 0.153974
  Epoch 60/100 — Loss: 0.112233
  Epoch 80/100 — Loss: 0.095821
  Epoch 100/100 — Loss: 0.092928
✅ Rice retrained!


In [81]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv", skiprows=1)
top_market = df['Market'].value_counts().index[0]
df = df[df['Market'] == top_market].copy()
print(df.columns.tolist())
print(df.head(3))

['State', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']
             State District        Market Commodity Group Commodity Variety  \
21633  West Bengal    Nadia  Kalyani APMC         Cereals      Rice    Fine   
21634  West Bengal    Nadia  Kalyani APMC         Cereals      Rice  Masuri   
21635  West Bengal    Nadia  Kalyani APMC         Cereals      Rice    Sona   

       Grade Min Price Max Price Modal Price   Price Unit  Price Date  
21633  Local  4,800.00  5,000.00    4,900.00  Rs./Quintal  27-04-2026  
21634  Local  3,300.00  3,500.00    3,400.00  Rs./Quintal  27-04-2026  
21635  Local  4,600.00  4,800.00    4,700.00  Rs./Quintal  27-04-2026  


In [82]:
commodity_files = {
    "Onion":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv",
    "Wheat":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv",
    "Cabbage":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv",
    "Cauliflower": "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv",
    "Brinjal":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv",
}

MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"

for commodity, filepath in commodity_files.items():
    print(f"\n{'='*50}")
    df = pd.read_csv(filepath, skiprows=1)
    top_market = df['Market'].value_counts().index[0]
    print(f"Training: {commodity} — Market: {top_market}")

    df = df[df['Market'] == top_market].copy()
    df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
    df = df.sort_values('Price Date')

    # Comma fix
    price_series = df['Modal Price'].astype(str).str.replace(',', '').pipe(pd.to_numeric, errors='coerce').dropna().values

    print(f"Rows: {len(price_series)}")
    print(f"Price range: ₹{price_series.min():.0f} – ₹{price_series.max():.0f} | Mean: ₹{price_series.mean():.0f}")

    if len(price_series) < 10:
        print(f"⚠ Not enough data, skipping.")
        continue

    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(price_series.reshape(-1, 1))

    SEQ_LEN = 3
    X, y = [], []
    for i in range(SEQ_LEN, len(scaled_data)):
        X.append(scaled_data[i-SEQ_LEN:i])
        y.append(scaled_data[i])

    X = torch.tensor(np.array(X, dtype=np.float32)).view(-1, SEQ_LEN, 1)
    y = torch.tensor(np.array(y, dtype=np.float32))

    model = LSTMModel()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(100):
        model.train()
        outputs = model(X)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/100 — Loss: {loss.item():.6f}")

    torch.save(model.state_dict(), f"{MODEL_DIR}{commodity.lower()}_lstm_model.pth")
    joblib.dump(scaler, f"{MODEL_DIR}{commodity.lower()}_scaler.save")
    print(f"✅ {commodity} done!")

print("\n🎉 All models retrained!")


Training: Onion — Market: Bangalore APMC
Rows: 498
Price range: ₹300 – ₹2200 | Mean: ₹1070
  Epoch 20/100 — Loss: 0.053512
  Epoch 40/100 — Loss: 0.051181
  Epoch 60/100 — Loss: 0.048846
  Epoch 80/100 — Loss: 0.041405
  Epoch 100/100 — Loss: 0.039681
✅ Onion done!

Training: Wheat — Market: Ganjbasoda APMC
Rows: 849
Price range: ₹1800 – ₹4350 | Mean: ₹2538
  Epoch 20/100 — Loss: 0.015351
  Epoch 40/100 — Loss: 0.015309
  Epoch 60/100 — Loss: 0.015303
  Epoch 80/100 — Loss: 0.015298
  Epoch 100/100 — Loss: 0.015294
✅ Wheat done!

Training: Cabbage — Market: Bowenpally APMC
Rows: 306
Price range: ₹370 – ₹1800 | Mean: ₹813
  Epoch 20/100 — Loss: 0.034146
  Epoch 40/100 — Loss: 0.024757
  Epoch 60/100 — Loss: 0.021395
  Epoch 80/100 — Loss: 0.020716
  Epoch 100/100 — Loss: 0.020158
✅ Cabbage done!

Training: Cauliflower — Market: RSPuram(Uzhavar Sandhai )
Rows: 182
Price range: ₹2750 – ₹8000 | Mean: ₹4347
  Epoch 20/100 — Loss: 0.025467
  Epoch 40/100 — Loss: 0.003795
  Epoch 60/100 — Lo

In [83]:
# Kaunse models purane hain check karo
import os
MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"
for f in os.listdir(MODEL_DIR):
    if f.endswith('.pth'):
        size = os.path.getsize(f"{MODEL_DIR}{f}")
        print(f"{f} — {size} bytes")

cauliflower_lstm_model.pth — 21321 bytes
onion_lstm_model.pth — 21185 bytes
cabbage_lstm_model.pth — 21209 bytes
rice_lstm_model.pth — 21173 bytes
wheat_lstm_model.pth — 21185 bytes
brinjal_lstm_model.pth — 21209 bytes


In [84]:
df = pd.read_csv("/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv", skiprows=1)
top_market = df['Market'].value_counts().index[0]
print(f"Market: {top_market}")

df = df[df['Market'] == top_market].copy()
df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
df = df.sort_values('Price Date')
price_series = df['Modal Price'].astype(str).str.replace(',', '').pipe(pd.to_numeric, errors='coerce').dropna().values

print(f"Rows: {len(price_series)}")
print(f"Price range: ₹{price_series.min():.0f} – ₹{price_series.max():.0f} | Mean: ₹{price_series.mean():.0f}")

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(price_series.reshape(-1, 1))

SEQ_LEN = 3
X, y = [], []
for i in range(SEQ_LEN, len(scaled_data)):
    X.append(scaled_data[i-SEQ_LEN:i])
    y.append(scaled_data[i])

X = torch.tensor(np.array(X, dtype=np.float32)).view(-1, SEQ_LEN, 1)
y = torch.tensor(np.array(y, dtype=np.float32))

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    outputs = model(X)
    loss = criterion(outputs, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1}/100 — Loss: {loss.item():.6f}")

MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"
torch.save(model.state_dict(), f"{MODEL_DIR}cauliflower_lstm_model.pth")
joblib.dump(scaler, f"{MODEL_DIR}cauliflower_scaler.save")
print("✅ Cauliflower retrained!")

Market: RSPuram(Uzhavar Sandhai )
Rows: 182
Price range: ₹2750 – ₹8000 | Mean: ₹4347
  Epoch 20/100 — Loss: 0.025295
  Epoch 40/100 — Loss: 0.004011
  Epoch 60/100 — Loss: 0.001779
  Epoch 80/100 — Loss: 0.001499
  Epoch 100/100 — Loss: 0.001450
✅ Cauliflower retrained!


In [85]:
commodity_files = {
    "Onion":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Onion.csv",
    "Rice":        "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Rice.csv",
    "Wheat":       "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Wheat.csv",
    "Cabbage":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cabbage.csv",
    "Cauliflower": "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/Cauliflower.csv",
    "Brinjal":     "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/data/brinjal.csv",
}
MODEL_DIR = "/Users/mukulbhamgare/Desktop/CommodityPricePrediction/app/models/"

for commodity, filepath in commodity_files.items():
    print(f"\n{'='*50}")
    df = pd.read_csv(filepath, skiprows=1)
    top_market = df['Market'].value_counts().index[0]
    print(f"Training: {commodity} — Market: {top_market}")
    df = df[df['Market'] == top_market].copy()
    df['Price Date'] = pd.to_datetime(df['Price Date'], dayfirst=True)
    df = df.sort_values('Price Date')
    price_series = df['Modal Price'].astype(str).str.replace(',', '').pipe(pd.to_numeric, errors='coerce').dropna().values
    print(f"Rows: {len(price_series)} | Price range: ₹{price_series.min():.0f} – ₹{price_series.max():.0f} | Mean: ₹{price_series.mean():.0f}")
    if len(price_series) < 10:
        print(f"⚠ Skipping.")
        continue
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(price_series.reshape(-1, 1))
    SEQ_LEN = 3
    X, y = [], []
    for i in range(SEQ_LEN, len(scaled_data)):
        X.append(scaled_data[i-SEQ_LEN:i])
        y.append(scaled_data[i])
    X = torch.tensor(np.array(X, dtype=np.float32)).view(-1, SEQ_LEN, 1)
    y = torch.tensor(np.array(y, dtype=np.float32))
    model = LSTMModel()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(100):
        model.train()
        outputs = model(X)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/100 — Loss: {loss.item():.6f}")
    torch.save(model.state_dict(), f"{MODEL_DIR}{commodity.lower()}_lstm_model.pth")
    joblib.dump(scaler, f"{MODEL_DIR}{commodity.lower()}_scaler.save")
    print(f"✅ {commodity} done!")

print("\n🎉 All models retrained!")


Training: Onion — Market: Bangalore APMC
Rows: 498 | Price range: ₹300 – ₹2200 | Mean: ₹1070
  Epoch 20/100 — Loss: 0.052514
  Epoch 40/100 — Loss: 0.051755
  Epoch 60/100 — Loss: 0.051142
  Epoch 80/100 — Loss: 0.050068
  Epoch 100/100 — Loss: 0.047411
✅ Onion done!

Training: Rice — Market: Kalyani APMC
Rows: 410 | Price range: ₹3300 – ₹4900 | Mean: ₹4316
  Epoch 20/100 — Loss: 0.173395
  Epoch 40/100 — Loss: 0.138654
  Epoch 60/100 — Loss: 0.097677
  Epoch 80/100 — Loss: 0.094452
  Epoch 100/100 — Loss: 0.092641
✅ Rice done!

Training: Wheat — Market: Ganjbasoda APMC
Rows: 849 | Price range: ₹1800 – ₹4350 | Mean: ₹2538
  Epoch 20/100 — Loss: 0.020523
  Epoch 40/100 — Loss: 0.015977
  Epoch 60/100 — Loss: 0.015364
  Epoch 80/100 — Loss: 0.015306
  Epoch 100/100 — Loss: 0.015293
✅ Wheat done!

Training: Cabbage — Market: Bowenpally APMC
Rows: 306 | Price range: ₹370 – ₹1800 | Mean: ₹813
  Epoch 20/100 — Loss: 0.034046
  Epoch 40/100 — Loss: 0.022665
  Epoch 60/100 — Loss: 0.020948
  